In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|               grupo|                item|valor|
+--------------------+---------+-------------------+--------------------+--------------------+-----+
|69d65b575eb6b44d5...|   Travel|2026-04-08 13:42:47|        G1_Ecommerce|  It's Only Himalaya|45.17|
|69d65b575eb6b44d5...|   Travel|2026-04-08 13:42:47|        G1_Ecommerce|Full Moon over No...|49.43|
|69defbab676143461...|   Travel|2026-04-15 02:44:59|G5_Turismo_y_Host...|  It's Only Himalaya|45.17|
|69defbab676143461...|   Travel|2026-04-15 02:44:59|G5_Turismo_y_Host...|Full Moon over No...|49.43|
|69df8180676143461...|   Travel|2026-04-15 12:16:00|G5_T

In [2]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|               grupo|                item|valor|
+--------------------+---------+-------------------+--------------------+--------------------+-----+
|69d65b575eb6b44d5...|   Travel|2026-04-08 13:42:47|        G1_Ecommerce|  It's Only Himalaya|45.17|
|69d65b575eb6b44d5...|   Travel|2026-04-08 13:42:47|        G1_Ecommerce|Full Moon over No...|49.43|
|69defbab676143461...|   Travel|2026-04-15 02:44:59|G5_Turismo_y_Host...|  It's Only Himalaya|45.17|
|69defbab676143461...|   Travel|2026-04-15 02:44:59|G5_Turismo_y_Host...|Full Moon over No...|49.43|
|69df8180676143461...|   Travel|2026-04-15 12:16:00|G5_T

In [3]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 6
Registros validos: 6


In [4]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
+--------------------+-----+



In [5]:
df.filter(df["valor"] > 100).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [6]:
df.groupBy("grupo").count().show()

+--------------------+-----+
|               grupo|count|
+--------------------+-----+
|G5_Turismo_y_Host...|    4|
|        G1_Ecommerce|    2|
+--------------------+-----+



In [7]:
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+------------------+-------------+-------------+
|categoria|cantidad_libros|   precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+------------------+-------------+-------------+
|   Travel|              6|47.300000000000004|        45.17|        49.43|
+---------+---------------+------------------+-------------+-------------+



In [8]:
import pyspark
print(pyspark.__version__)

3.5.0


In [9]:
import pyspark
print("Spark version:", pyspark.__version__)

# Ver qué JARs están cargados actualmente
sc = spark.sparkContext
print("JARs activos:")
for jar in sc._conf.get("spark.jars", "").split(","):
    print(" -", jar)

# También revisa el classpath
import subprocess
result = subprocess.run(["find", "/", "-name", "mongo-spark-connector*.jar", "-not", "-path", "*/proc/*"], 
                       capture_output=True, text=True, timeout=10)
print("JARs de mongo encontrados:")
print(result.stdout)

Spark version: 3.5.0
JARs activos:
 - file:///home/jovyan/.ivy2/jars/org.mongodb.spark_mongo-spark-connector_2.12-10.3.0.jar
 - file:///home/jovyan/.ivy2/jars/org.mongodb_mongodb-driver-sync-4.8.2.jar
 - file:///home/jovyan/.ivy2/jars/org.mongodb_bson-4.8.2.jar
 - file:///home/jovyan/.ivy2/jars/org.mongodb_mongodb-driver-core-4.8.2.jar
 - file:///home/jovyan/.ivy2/jars/org.mongodb_bson-record-codec-4.8.2.jar
JARs de mongo encontrados:
/home/jovyan/.ivy2/cache/org.mongodb.spark/mongo-spark-connector_2.12/jars/mongo-spark-connector_2.12-3.0.2.jar
/home/jovyan/.ivy2/cache/org.mongodb.spark/mongo-spark-connector_2.12/jars/mongo-spark-connector_2.12-10.3.0.jar
/home/jovyan/.ivy2/cache/org.mongodb.spark/mongo-spark-connector_2.12/jars/mongo-spark-connector_2.12-10.1.1.jar



In [11]:
from pyspark.sql import functions as F

 # 1. Ordenar por valor de forma descendente
 # 2. Tomar el primer registro
ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("item", "valor", "grupo", "fecha_captura").show()

+--------------------+-----+------------+-------------------+
|                item|valor|       grupo|      fecha_captura|
+--------------------+-----+------------+-------------------+
|Full Moon over No...|49.43|G1_Ecommerce|2026-04-08 13:42:47|
+--------------------+-----+------------+-------------------+



In [19]:
NOMBRE_GRUPO = "G5_Turismo_y_Hosteleria"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        # tus agregaciones aquí
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: G5_Turismo_y_Hosteleria ---
+---------+---------------------+
|categoria|ultima_sincronizacion|
+---------+---------------------+
|   Travel|  2026-04-15 12:16:00|
+---------+---------------------+

